# Lesson 07 - Tervezési mintázat

Ez a jegyzetfüzet bemutatja az **AI ügynökök tervezési mintázatát** a Microsoft Agent Framework segítségével. Megtanulod, hogyan bontsd le az összetett utazási kéréseket strukturált alfeladatokra, hogyan oszd ki őket szakértő ügynököknek, és hogyan hajtsd végre az így létrejövő tervet — mindezt Pydantic modelleken alapuló strukturált kimenet segítségével.


## Beállítás


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os, asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Feladat lebontása

A feladat lebontása a tervezési minta magja. Ahelyett, hogy egyetlen ügynököt kérnénk meg egy összetett kérés végponttól végpontig történő kezelésére, a problémát kisebb, jól körülhatárolt **al-feladatokra** bontjuk. Minden al-feladat egy specialistának van kiosztva (pl. járatok, szállodák, programok) világos prioritásokkal és függőségi sorrenddel.

Ez a megközelítés több előnnyel jár:
- **Átláthatóság**: minden al-feladatnak egyetlen felelőssége van.
- **Párhuzamosság**: független al-feladatok párhuzamosan futtathatók.
- **Megbízhatóság**: a hibák egyedi al-feladatokra korlátozódnak.
- **Költségkövetés**: a költségek al-feladatonként becsülhetők és összesíthetők.


In [ ]:
class TravelSubTask(BaseModel):
    task_id: int
    description: str
    assigned_agent: str  # "flight_agent", "hotel_agent", "activity_agent"
    priority: str  # "high", "medium", "low"
    dependencies: list[int] = []


class TravelPlan(BaseModel):
    destination: str
    trip_duration_days: int
    subtasks: list[TravelSubTask]
    total_estimated_budget_usd: int
    notes: str

## Egy tervezési ügynök létrehozása strukturált kimenettel

A tervezési ügynök mint **portaszolgálati koordinátor** működik. Egy magas szintű utazási kérés alapján strukturált `TravelPlan`-t készít — lebontva a kérést részfeladatokra, prioritásokat állít be, és azonosítja a függőségeket, hogy a concierge vagy a végrehajtó réteg elvégezhesse a munkát.


In [ ]:
planning_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="""You are a travel planning agent. When given a travel request:
1. Break it into specific subtasks (flights, hotels, activities, logistics)
2. Assign each subtask to the appropriate specialist agent
3. Set priorities and identify dependencies between tasks
4. Estimate the total budget""",
)

result = await planning_agent.run(
    "Plan a 7-day trip to Paris for a couple interested in art, cuisine, and history. Budget around $5000.",
    )
if result:
    plan = result
    print(f"Destination: {plan.destination}")
    print(f"Duration: {plan.trip_duration_days} days")
    print(f"Budget: ${plan.total_estimated_budget_usd}")
    print(f"\nSubtasks:")
    for task in plan.subtasks:
        print(f"  [{task.priority}] {task.task_id}. {task.description} → {task.assigned_agent}")

## Egy terv végrehajtása szakértői eszközökkel

Miután a recepciós ügynök elkészítette a strukturált tervet, a **konzjerzs ügynök** hajtja végre azt.
Minden szakértői eszköz egy-egy részfeladat-kategóriát kezel (repülések, szállodák, programok). A konzjerzs
a terv részfeladatain átfut azok függőségi sorrendjében, és mindegyiket a megfelelő eszköznek továbbítja.


In [ ]:
@tool
def book_flight(
    destination: Annotated[str, "The destination city"],
    departure_date: Annotated[str, "Departure date (YYYY-MM-DD)"],
    return_date: Annotated[str, "Return date (YYYY-MM-DD)"],
) -> str:
    """Search and book flights for the trip."""
    return f"Flight booked to {destination}: {departure_date} → {return_date}, confirmation #FLT-{hash(destination) % 10000:04d}"


@tool
def reserve_hotel(
    city: Annotated[str, "The city for the hotel"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"],
) -> str:
    """Reserve a hotel room in the destination city."""
    return f"Hotel reserved in {city}: {check_in} to {check_out} for {guests} guests, confirmation #HTL-{hash(city) % 10000:04d}"


@tool
def book_activity(
    activity_name: Annotated[str, "Name of the activity or tour"],
    date: Annotated[str, "Date of the activity (YYYY-MM-DD)"],
    participants: Annotated[int, "Number of participants"],
) -> str:
    """Book a tour, museum visit, or other activity."""
    return f"Activity booked: {activity_name} on {date} for {participants} people, confirmation #ACT-{hash(activity_name) % 10000:04d}"


# Concierge agent that executes the plan using specialist tools
concierge_agent = await provider.create_agent(
    name="Concierge",
    instructions="""You are a travel concierge executing a structured travel plan.
Use the available tools to fulfil each subtask. Work through the subtasks in order,
respecting dependencies. Summarise the results when finished.""",
    tools=[book_flight, reserve_hotel, book_activity],
)

# Build a prompt from the plan produced above
if result.value:
    subtask_lines = "\n".join(
        f"- [{t.priority}] {t.task_id}. {t.description} (agent: {t.assigned_agent}, deps: {t.dependencies})"
        for t in plan.subtasks
    )
    execution_prompt = (
        f"Execute the following travel plan for {plan.destination} "
        f"({plan.trip_duration_days} days, ${plan.total_estimated_budget_usd} budget):\n"
        f"{subtask_lines}"
    )

    exec_response = await concierge_agent.run(execution_prompt)
    print(exec_response)

## Összefoglaló

Ebben a leckében megismerted az AI ügynökök **Tervezési mintáját**:

1. **Feladat lebontás** — Egy recepciós tervező ügynök összetett utazási kérést bont le strukturált alfeladatokra Pydantic modellek segítségével, mindegyiket egy szakértői ügynökhöz rendelve prioritásokkal és függőségekkel.
2. **Strukturált kimenet** — A `response_format` megadásával az ügynök egy érvényesített `TravelPlan` objektumot ad vissza a szabad szöveg helyett, így a további feldolgozás megbízható lesz.
3. **Terv végrehajtás** — Egy concierge ügynök végigiterál az alfeladatokon, szakértői eszközöket (`book_flight`, `reserve_hotel`, `book_activity`) használva a terv végrehajtásához és az eredmények jelentéséhez.

Ez a minta szétválasztja a *mit kell tenni* (tervezés) a *hogyan kell tenni* (végrehajtás) részét, így modulárisabbá, tesztelhetőbbé és könnyebben bővíthetővé teszi az ügynököket.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Jogi nyilatkozat**:
Ez a dokumentum az AI fordító szolgáltatás, a [Co-op Translator](https://github.com/Azure/co-op-translator) segítségével készült. Bár a pontosságra törekszünk, kérjük, vegye figyelembe, hogy az automatikus fordítások hibákat vagy pontatlanságokat tartalmazhatnak. Az eredeti dokumentum az anyanyelvén tekintendő hiteles forrásnak. Kritikus információk esetén javasoljuk szakmai, emberi fordító igénybevételét. Nem vállalunk felelősséget a fordítás használatából eredő félreértésekért vagy helytelen értelmezésekért.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
